In [5]:
# Human-in-the-Loop Review Map

import geopandas as gpd
import folium
import branca.colormap as cm

print("Building Human-in-the-Loop review map...\n")

# Load file
df = gpd.read_file("compliance_results_ultimate_final.geojson")

# Add stable ID
df["structure_id"] = "S" + df.index.astype(str).str.zfill(4)

# Reproject for web map
df_web = df.to_crs("EPSG:4326")

# Base map
m = folium.Map(location=[-34.04, 18.68], zoom_start=14, tiles="CartoDB positron")

# Colormap
colormap = cm.LinearColormap(
    colors=['#00ff00', '#ffff00', '#ff9900', '#ff0000'],
    vmin=0.0, vmax=1.0,
    caption='Final Risk Score (0-1)'
).add_to(m)

# Feature groups for filtering
critical_layer = folium.FeatureGroup(name="🔴 Critical Risk")
high_layer = folium.FeatureGroup(name="🟠 High Risk")
moderate_layer = folium.FeatureGroup(name="🟡 Moderate / Low")

# Add polygons
for _, row in df_web.iterrows():
    sid = row["structure_id"]
    color = colormap(row['final_risk_score'])
    
    popup_html = f"""
    <div style="font-family: Arial; width: 340px;">
        <h4>Structure ID: {sid}</h4>
        <b>Area:</b> {row['area_m2']:.1f} m²<br>
        <b>Zoning:</b> {row.get('INT_ZONE_DESC', 'N/A')}<br>
        <b>Status:</b> {row['compliance_status']}<br>
        <b>Risk:</b> {row['final_risk_score']:.2f} ({row['risk_category']})<br>
        <b>Growth Type:</b> {row['growth_type']}<br>
        <b>Dist to Road:</b> {row.get('dist_to_road_m', 'N/A'):.1f} m<br>
        <b>Dist to Existing:</b> {row.get('dist_to_existing_m', 'N/A'):.1f} m<br>
        <hr>
        <b>Reviewer Decision:</b><br>
        <button onclick="saveDecision('{sid}', 'Valid')" style="background:#28a745;color:white;padding:8px 12px;margin:3px;border:none;border-radius:4px;">✅ Valid</button>
        <button onclick="saveDecision('{sid}', 'Illegal')" style="background:#dc3545;color:white;padding:8px 12px;margin:3px;border:none;border-radius:4px;">🚫 Illegal</button>
        <button onclick="saveDecision('{sid}', 'Uncertain')" style="background:#ffc107;color:black;padding:8px 12px;margin:3px;border:none;border-radius:4px;">❓ Uncertain</button>
        <br><br>
        <b>Current Decision:</b> <span id="dec-{sid}" style="font-weight:bold;">None</span>
    </div>
    """

    feature = folium.GeoJson(
        row.geometry.__geo_interface__,
        style_function=lambda x, c=color: {'fillColor': c, 'color': 'black', 'weight': 1.5, 'fillOpacity': 0.75},
        popup=folium.Popup(popup_html, max_width=380)
    )

    if row['risk_category'] == "Critical":
        feature.add_to(critical_layer)
    elif row['risk_category'] == "High":
        feature.add_to(high_layer)
    else:
        feature.add_to(moderate_layer)

# Add layers
critical_layer.add_to(m)
high_layer.add_to(m)
moderate_layer.add_to(m)

# JavaScript for decisions and summary
js = """
<script>
let decisions = {};

function saveDecision(id, decision) {
    decisions[id] = decision;
    let el = document.getElementById("dec-" + id);
    if (el) el.innerText = decision;
    updateSummary();
    
    // Auto-save to localStorage
    localStorage.setItem("review_decisions", JSON.stringify(decisions));
}

function updateSummary() {
    let valid = 0, illegal = 0, uncertain = 0;
    Object.values(decisions).forEach(d => {
        if (d === "Valid") valid++;
        if (d === "Illegal") illegal++;
        if (d === "Uncertain") uncertain++;
    });
    let html = `<b>Review Summary</b><br>
                ✅ Valid: ${valid}<br>
                🚫 Illegal: ${illegal}<br>
                ❓ Uncertain: ${uncertain}<br>
                Total Reviewed: ${Object.keys(decisions).length}`;
    let panel = document.getElementById("summary-panel");
    if (panel) panel.innerHTML = html;
}

// Load previous decisions on page load
window.onload = function() {
    let saved = localStorage.getItem("review_decisions");
    if (saved) {
        decisions = JSON.parse(saved);
        Object.keys(decisions).forEach(id => {
            let el = document.getElementById("dec-" + id);
            if (el) el.innerText = decisions[id];
        });
        updateSummary();
    }
};
</script>
"""
m.get_root().html.add_child(folium.Element(js))

# Floating summary panel
summary_html = """
<div id="summary-panel" style="position:fixed;bottom:20px;right:20px;background:white;padding:12px;border:2px solid #333;border-radius:8px;z-index:1000;font-size:14px;box-shadow:0 2px 10px rgba(0,0,0,0.3);">
<b>Review Summary</b><br>Loading...
</div>
"""
m.get_root().html.add_child(folium.Element(summary_html))

# Controls
folium.LayerControl(collapsed=False).add_to(m)
colormap.add_to(m)

# Save map
m.save("compliance_review_map_final.html")
print("Review map saved as: compliance_review_map_final.html")

# Display
m

Building Human-in-the-Loop review map...

Review map saved as: compliance_review_map_final.html
